# Fit example: \(D^+\to\pi^-\pi^+\pi^+\)

This notebook demonstrates a real unbinned amplitude fit with weighted `phasespace` Monte Carlo, automatic identical-particle symmetrization, `RealImag` coefficients, floating resonance mass and width, JAX gradients, and `iminuit`.

A simple \(\rho(770)^0+\) constant non-resonant model is used so the fit mechanics are easy to inspect.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel, DecayModel, Minimizer, NonResonant,
    Parameter, RealImag, Resonance, enable_x64, weighted_resample,
)

enable_x64()


## 1. Declare channel and fit parameters

The non-resonant coefficient is fixed to \(1+0i\), defining the global amplitude scale and phase. The fit floats

\[
x_\rho,\quad y_\rho,\quad m_\rho,\quad \Gamma_\rho.
\]


In [ ]:
rho_x = Parameter.coefficient(
    "rho.x", 0.55, owner="rho",
    bounds=(-1.5, 1.5), step=0.02,
)
rho_y = Parameter.coefficient(
    "rho.y", -0.10, owner="rho",
    bounds=(-1.5, 1.5), step=0.02,
)
rho_mass = Parameter.dynamics(
    "rho.mass", 0.755, owner="rho",
    bounds=(0.73, 0.81), step=0.001,
)
rho_width = Parameter.dynamics(
    "rho.width", 0.180, owner="rho",
    bounds=(0.10, 0.22), step=0.002,
)

channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

model = DecayModel(
    channel,
    [
        Resonance(
            "rho",
            pair=(0, 1),
            coefficient=RealImag(rho_x, rho_y),
            mass=rho_mass,
            width=rho_width,
            spin=1,
        ),
        NonResonant(RealImag(1.0, 0.0)),
    ],
)

model.parameters


## 2. Injected truth and displaced starting point


In [ ]:
truth = {
    "rho.x": 0.85,
    "rho.y": 0.30,
    "rho.mass": 0.775,
    "rho.width": 0.149,
}
start = {parameter.name: parameter.value for parameter in model.parameters}

print("start:", start)
print("truth:", truth)


## 3. Generate unweighted pseudo-data

For phase-space candidates \(x_k\),

\[
w_k^{target}=w_k^{PS}|A(x_k;\theta_{gen})|^2.
\]

We generate 1,000,000 candidates and resample 100,000 unweighted pseudo-events.


In [ ]:
N_CANDIDATES = 1_000_000
N_DATA = 100_000

candidate_pool = model.generate_phase_space(N_CANDIDATES, seed=2026)
truth_intensity = model.intensity(candidate_pool.as_dict(), truth)
target_weights = candidate_pool.weights * truth_intensity

data = weighted_resample(
    jax.random.key(2026),
    candidate_pool,
    target_weights,
    N_DATA,
    replace=True,
)

print("candidate events:", candidate_pool.size)
print("pseudo-data events:", data.size)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
h = ax.hist2d(np.asarray(data.s12), np.asarray(data.s13), bins=100)
fig.colorbar(h[3], ax=ax, label="events")
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title("Generated pseudo-data")
plt.show()


## 4. Independent 1M-event normalization sample


In [ ]:
N_NORM = 1_000_000
normalization_sample = model.generate_phase_space(N_NORM, seed=2027)
cache = model.prepare_cache(data, normalization_sample)
print("normalization events:", normalization_sample.size)


## 5. Unbinned negative log-likelihood

\[
-\log\mathcal L =
-\sum_n \log |A(x_n)|^2
+N_{data}\log\mathcal N(\theta).
\]


In [ ]:
def nll(values):
    intensity, normalization = cache.evaluate(values)
    return (
        -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300)))
        + data.size * jnp.log(normalization)
    )

print("NLL(start):", float(nll(start)))
print("NLL(truth):", float(nll(truth)))


## 6. Fit with Minuit


In [ ]:
minimizer = Minimizer(nll, model.parameters)
result = minimizer.fit()

print(result)
print("valid:", result.valid)
print("fval:", result.fval)


## 7. Closure pulls

For each free parameter,

\[
pull=\frac{x_{gen}-x_{fit}}{\sigma_{fit}}.
\]

The project closure convention is \(|pull|<1\).


In [ ]:
fit_values = {name: float(result.values[name]) for name in result.parameters}

rows = []
for parameter in model.parameters:
    if parameter.fixed:
        continue
    name = parameter.name
    generated = truth[name]
    fitted = float(result.values[name])
    error = float(result.errors[name])
    pull = (generated - fitted) / error
    rows.append((name, generated, fitted, error, pull, abs(pull) < 1.0))

print(f"{'parameter':12s} {'gen':>10s} {'fit':>10s} {'error':>10s} {'pull':>10s} {'<1 sigma':>10s}")
for name, generated, fitted, error, pull, compatible in rows:
    print(
        f"{name:12s} {generated:10.5f} {fitted:10.5f} "
        f"{error:10.5f} {pull:10.3f} {str(compatible):>10s}"
    )


## 8. Projection before and after the fit


In [ ]:
def projection_hist(values, bins):
    weights = np.asarray(
        normalization_sample.weights
        * model.intensity(normalization_sample.as_dict(), values)
    )
    h12, _ = np.histogram(np.asarray(normalization_sample.s12), bins=bins, weights=weights)
    h13, _ = np.histogram(np.asarray(normalization_sample.s13), bins=bins, weights=weights)
    return h12 + h13

s_data = np.concatenate([np.asarray(data.s12), np.asarray(data.s13)])
bins = np.linspace(s_data.min(), s_data.max(), 100)
centers = 0.5 * (bins[:-1] + bins[1:])

data_hist, _ = np.histogram(s_data, bins=bins)
start_hist = projection_hist(start, bins)
fit_hist = projection_hist(fit_values, bins)

start_hist *= data_hist.sum() / start_hist.sum()
fit_hist *= data_hist.sum() / fit_hist.sum()

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(
    centers, data_hist,
    yerr=np.sqrt(np.maximum(data_hist, 1)),
    fmt=".", label="pseudo-data",
)
ax.step(centers, start_hist, where="mid", label="before fit")
ax.step(centers, fit_hist, where="mid", label="after fit")
ax.set_xlabel(r"$m^2(\pi^-\pi^+)$ [GeV$^2$]")
ax.set_ylabel("entries / bin")
ax.legend()
ax.set_title("Projection: data vs model")
plt.show()


## 9. Fitted Dalitz density


In [ ]:
fit_weights = np.asarray(
    normalization_sample.weights
    * model.intensity(normalization_sample.as_dict(), fit_values)
)

fig, ax = plt.subplots(figsize=(7, 6))
h = ax.hist2d(
    np.asarray(normalization_sample.s12),
    np.asarray(normalization_sample.s13),
    bins=100,
    weights=fit_weights,
)
fig.colorbar(h[3], ax=ax, label=r"$w_{PS}|A_{fit}|^2$")
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title("Fitted Dalitz density")
plt.show()
